## 환경설정

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import koreanize_matplotlib
from sqlalchemy import create_engine

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

In [ ]:
from dotenv import load_dotenv
load_dotenv()

engine = create_engine(os.environ["DB_URL"])
conn = engine.connect()

In [ ]:
DATE_FMT = "%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f"

RETENTION_START_HOUR = 24
RETENTION_WINDOW_DAY = 7

In [ ]:
def _normalize_sql(sql: str) -> str:

    return sql.replace("%%", "%")


def run_query(query: str, name: str | None = None) -> pd.DataFrame:
    result = conn.exec_driver_sql(_normalize_sql(query))
    rows = result.fetchall()
    df = pd.DataFrame(rows, columns=result.keys())
    if name:
        print(f"[{name}] rows={len(df):,}, cols={len(df.columns):,}")
    return df


def execute_many(sql: str) -> None:
    statements = [stmt.strip() for stmt in sql.split(";") if stmt.strip()]
    for stmt in statements:
        conn.exec_driver_sql(_normalize_sql(stmt))
    try:
        conn.commit()
    except Exception:
        pass
    print(f"Executed {len(statements):,} statements.")

## 데이터 전처리

### VIEW 생성 및 결측값 제거

- 이 과정에서 user_id 결측 + event_time 변환 실패 행 제거

In [ ]:
create_views_sql = """

DROP VIEW IF EXISTS v_events_signup;
CREATE VIEW v_events_signup AS
SELECT
    user_id,
    STR_TO_DATE(event_ts, '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f') AS event_time
FROM events_signup
WHERE user_id IS NOT NULL
  AND user_id <> ''
  AND STR_TO_DATE(event_ts, '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f') IS NOT NULL;


DROP VIEW IF EXISTS v_events_content_start;
CREATE VIEW v_events_content_start AS
SELECT
    user_id,
    STR_TO_DATE(event_ts, '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f') AS event_time,
    `content_id` AS content_id
FROM events_content_start
WHERE user_id IS NOT NULL
  AND user_id <> ''
  AND STR_TO_DATE(event_ts, '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f') IS NOT NULL;


DROP VIEW IF EXISTS v_events_lesson_view;
CREATE VIEW v_events_lesson_view AS
SELECT
    user_id,
    STR_TO_DATE(event_ts, '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f') AS event_time,
    `content_id` AS content_id,
    `lesson_id`  AS lesson_id
FROM events_lesson_view
WHERE user_id IS NOT NULL
  AND user_id <> ''
  AND STR_TO_DATE(event_ts, '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f') IS NOT NULL;


DROP VIEW IF EXISTS v_events_lesson_complete;
CREATE VIEW v_events_lesson_complete AS
SELECT
    user_id,
    STR_TO_DATE(event_ts, '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f') AS event_time,
    `content_id` AS content_id,
    `lesson_id`  AS lesson_id
FROM events_lesson_complete
WHERE user_id IS NOT NULL
  AND user_id <> ''
  AND STR_TO_DATE(event_ts, '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f') IS NOT NULL;


DROP VIEW IF EXISTS v_events_content_end;
CREATE VIEW v_events_content_end AS
SELECT
    user_id,
    STR_TO_DATE(event_ts, '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f') AS event_time,
    `content_id` AS content_id
FROM events_content_end
WHERE user_id IS NOT NULL
  AND user_id <> ''
  AND STR_TO_DATE(event_ts, '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f') IS NOT NULL;


DROP VIEW IF EXISTS v_events_related_question_click;
CREATE VIEW v_events_related_question_click AS
SELECT
    user_id,
    STR_TO_DATE(event_ts, '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f') AS event_time,
    `content_id`  AS content_id,
    `lesson_id`   AS lesson_id
FROM events_related_question_click
WHERE user_id IS NOT NULL
  AND user_id <> ''
  AND STR_TO_DATE(event_ts, '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f') IS NOT NULL;
"""

execute_many(create_views_sql)

### VIEW 생성 여부 검증

In [ ]:
run_query("""
SELECT 'v_events_signup'        AS view_name, COUNT(*) AS row_cnt FROM v_events_signup
UNION ALL SELECT 'v_events_content_start',           COUNT(*) FROM v_events_content_start
UNION ALL SELECT 'v_events_lesson_view',       COUNT(*) FROM v_events_lesson_view
UNION ALL SELECT 'v_events_lesson_complete',         COUNT(*) FROM v_events_lesson_complete
UNION ALL SELECT 'v_events_content_end',             COUNT(*) FROM v_events_content_end
UNION ALL SELECT 'v_events_related_question_click',  COUNT(*) FROM v_events_related_question_click;
""", "view_check")

### 결측값 확인

In [ ]:
DATE_FMT = '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f'

null_check_df = run_query(f"""
SELECT 'events_signup' AS table_name,
    COUNT(*) AS total,
    SUM(CASE WHEN NULLIF(TRIM(user_id), '') IS NULL THEN 1 ELSE 0 END) AS user_id_null,
    SUM(CASE WHEN STR_TO_DATE(event_ts, '{DATE_FMT}') IS NULL THEN 1 ELSE 0 END) AS event_time_null
FROM events_signup
UNION ALL
SELECT 'events_content_start', COUNT(*),
    SUM(CASE WHEN NULLIF(TRIM(user_id), '') IS NULL THEN 1 ELSE 0 END),
    SUM(CASE WHEN STR_TO_DATE(event_ts, '{DATE_FMT}') IS NULL THEN 1 ELSE 0 END)
FROM events_content_start
UNION ALL
SELECT 'events_lesson_view', COUNT(*),
    SUM(CASE WHEN NULLIF(TRIM(user_id), '') IS NULL THEN 1 ELSE 0 END),
    SUM(CASE WHEN STR_TO_DATE(event_ts, '{DATE_FMT}') IS NULL THEN 1 ELSE 0 END)
FROM events_lesson_view
UNION ALL
SELECT 'events_lesson_complete', COUNT(*),
    SUM(CASE WHEN NULLIF(TRIM(user_id), '') IS NULL THEN 1 ELSE 0 END),
    SUM(CASE WHEN STR_TO_DATE(event_ts, '{DATE_FMT}') IS NULL THEN 1 ELSE 0 END)
FROM events_lesson_complete
UNION ALL
SELECT 'events_content_end', COUNT(*),
    SUM(CASE WHEN NULLIF(TRIM(user_id), '') IS NULL THEN 1 ELSE 0 END),
    SUM(CASE WHEN STR_TO_DATE(event_ts, '{DATE_FMT}') IS NULL THEN 1 ELSE 0 END)
FROM events_content_end
UNION ALL
SELECT 'events_related_question_click', COUNT(*),
    SUM(CASE WHEN NULLIF(TRIM(user_id), '') IS NULL THEN 1 ELSE 0 END),
    SUM(CASE WHEN STR_TO_DATE(event_ts, '{DATE_FMT}') IS NULL THEN 1 ELSE 0 END)
FROM events_related_question_click;
""", "null_check")

null_check_df

### 중복값 확인

In [ ]:
duplicate_check_df = run_query("""
SELECT 'events_signup' AS table_name,
    COUNT(*) AS total,
    COUNT(DISTINCT user_id, event_time) AS unique_cnt,
    COUNT(*) - COUNT(DISTINCT user_id, event_time) AS duplicated_cnt
FROM v_events_signup
UNION ALL
SELECT 'events_content_start', COUNT(*), COUNT(DISTINCT user_id, event_time),
    COUNT(*) - COUNT(DISTINCT user_id, event_time)
FROM v_events_content_start
UNION ALL
SELECT 'events_lesson_view', COUNT(*), COUNT(DISTINCT user_id, event_time),
    COUNT(*) - COUNT(DISTINCT user_id, event_time)
FROM v_events_lesson_view
UNION ALL
SELECT 'events_lesson_complete', COUNT(*), COUNT(DISTINCT user_id, event_time),
    COUNT(*) - COUNT(DISTINCT user_id, event_time)
FROM v_events_lesson_complete
UNION ALL
SELECT 'events_content_end', COUNT(*), COUNT(DISTINCT user_id, event_time),
    COUNT(*) - COUNT(DISTINCT user_id, event_time)
FROM v_events_content_end
UNION ALL
SELECT 'events_related_question_click', COUNT(*),
    COUNT(DISTINCT user_id, event_time),
    COUNT(*) - COUNT(DISTINCT user_id, event_time)
FROM v_events_related_question_click;
""", "duplicate_check")

duplicate_check_df

### 이상치 1 : 시간 범위

In [ ]:
time_range_df = run_query("""
SELECT 'v_events_signup' AS view_name,
    MIN(event_time) AS min_t, MAX(event_time) AS max_t,
    SUM(CASE WHEN event_time > NOW() THEN 1 ELSE 0 END) AS future_cnt
FROM v_events_signup
UNION ALL
SELECT 'v_events_content_start', MIN(event_time), MAX(event_time),
    SUM(CASE WHEN event_time > NOW() THEN 1 ELSE 0 END)
FROM v_events_content_start
UNION ALL
SELECT 'v_events_lesson_view', MIN(event_time), MAX(event_time),
    SUM(CASE WHEN event_time > NOW() THEN 1 ELSE 0 END)
FROM v_events_lesson_view
UNION ALL
SELECT 'v_events_lesson_complete', MIN(event_time), MAX(event_time),
    SUM(CASE WHEN event_time > NOW() THEN 1 ELSE 0 END)
FROM v_events_lesson_complete
UNION ALL
SELECT 'v_events_content_end', MIN(event_time), MAX(event_time),
    SUM(CASE WHEN event_time > NOW() THEN 1 ELSE 0 END)
FROM v_events_content_end
UNION ALL
SELECT 'v_events_related_question_click', MIN(event_time), MAX(event_time),
    SUM(CASE WHEN event_time > NOW() THEN 1 ELSE 0 END)
FROM v_events_related_question_click;
""", "time_range")

time_range_df

### 이상치 2 : 가입 전 활동 (정합성)

In [ ]:
before_signup_df = run_query("""
WITH signup AS (
    SELECT user_id, MIN(event_time) AS signup_time
    FROM v_events_signup GROUP BY user_id
)
SELECT 'v_events_content_start' AS view_name,
    COUNT(*) AS before_signup_rows
FROM v_events_content_start sc
JOIN signup s ON sc.user_id = s.user_id
WHERE sc.event_time < s.signup_time
UNION ALL
SELECT 'v_events_lesson_view', COUNT(*)
FROM v_events_lesson_view l
JOIN signup s ON l.user_id = s.user_id
WHERE l.event_time < s.signup_time
UNION ALL
SELECT 'v_events_lesson_complete', COUNT(*)
FROM v_events_lesson_complete cl
JOIN signup s ON cl.user_id = s.user_id
WHERE cl.event_time < s.signup_time
UNION ALL
SELECT 'v_events_content_end', COUNT(*)
FROM v_events_content_end ec
JOIN signup s ON ec.user_id = s.user_id
WHERE ec.event_time < s.signup_time
UNION ALL
SELECT 'v_events_related_question_click', COUNT(*)
FROM v_events_related_question_click cq
JOIN signup s ON cq.user_id = s.user_id
WHERE cq.event_time < s.signup_time;
""", "before_signup")

before_signup_df

### 이상치 3 : 봇 의심 (한 유저가 너무 많은 이벤트)

- 유저별 일평균 활동량 분포 살펴보기

In [ ]:
user_stats_query = """
    SELECT
        user_id,
        COUNT(*) AS event_cnt,
        TIMESTAMPDIFF(DAY, MIN(event_time), MAX(event_time)) AS active_days,
        COUNT(*) / GREATEST(TIMESTAMPDIFF(DAY, MIN(event_time), MAX(event_time)), 1) AS daily_avg
    FROM v_events_lesson_view
    GROUP BY user_id
"""
df_user_stats = run_query(user_stats_query)

In [ ]:
# 기본 통계
print("전체 유저 수:", len(df_user_stats))
print("\n=== daily_avg 분포 ===")
print(df_user_stats['daily_avg'].describe(percentiles=[0.5, 0.9, 0.95, 0.99, 0.999]))

# 임계값별 잘리는 유저 수 시뮬레이션
print("\n=== 임계값별 봇으로 분류되는 유저 수 ===")
for threshold in [20, 30, 50, 100, 200, 500, 974]:
    bot_count = (df_user_stats['daily_avg'] >= threshold).sum()
    pct = bot_count / len(df_user_stats) * 100
    print(f"  ≥ {threshold:>4}/일 : {bot_count:>6}명 ({pct:.3f}%)")

In [ ]:
top_suspects = df_user_stats[df_user_stats['daily_avg'] >= 500].sort_values('daily_avg', ascending=False)
print(f"500/일 이상 유저: {len(top_suspects)}명")
print("\n=== 상위 20명 ===")
print(top_suspects[['user_id', 'event_cnt', 'active_days', 'daily_avg']].head(20))

print("\n=== 일평균 분포 (500+) ===")
print(top_suspects['daily_avg'].describe())

In [ ]:
sample_user = 'SAMPLE_USER_ID'

run_query(f"""
SELECT
    user_id,
    event_time,
    lesson_id,
    COUNT(*) AS dup_cnt
FROM v_events_lesson_view
WHERE user_id = '{sample_user}'
GROUP BY user_id, event_time, lesson_id
ORDER BY dup_cnt DESC
LIMIT 5;
""", "active0_check")

In [ ]:
THRESHOLD = 200
bot_candidates = df_user_stats[df_user_stats['daily_avg'] >= THRESHOLD].copy()
print(f"봇 후보: {len(bot_candidates)}명")

execute_many("""
DROP TABLE IF EXISTS bot_users;
CREATE TABLE bot_users (
    user_id     VARCHAR(64) NOT NULL PRIMARY KEY,
    event_cnt   INT,
    active_days INT,
    daily_avg   DECIMAL(10,2),
    detected_at DATETIME DEFAULT CURRENT_TIMESTAMP
) ENGINE=InnoDB;
""")

bot_candidates[['user_id', 'event_cnt', 'active_days', 'daily_avg']].to_sql(
    'bot_users', con=engine, if_exists='append', index=False
)

run_query("SELECT COUNT(*) AS cnt FROM bot_users", "bot_count")

## Revenue

### Revenue용 View 추가 생성

In [ ]:
DATE_FMT = '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f'

create_revenue_views_sql = f"""
DROP VIEW IF EXISTS v_events_payment_page_view;
CREATE VIEW v_events_payment_page_view AS
SELECT
    epp.user_id,
    STR_TO_DATE(epp.event_ts, '{DATE_FMT}') AS event_time
FROM events_payment_page_view epp
LEFT JOIN bot_users b ON epp.user_id = b.user_id
WHERE epp.user_id IS NOT NULL AND epp.user_id <> ''
  AND STR_TO_DATE(epp.event_ts, '{DATE_FMT}') IS NOT NULL
  AND b.user_id IS NULL;


DROP VIEW IF EXISTS v_events_subscription_complete;
CREATE VIEW v_events_subscription_complete AS
SELECT
    cs.user_id,
    STR_TO_DATE(cs.event_ts, '{DATE_FMT}') AS event_time,
    cs.paid_amount,
    cs.`plan_price`              AS plan_price,
    cs.`discount_amount`  AS coupon_discount_amount,
    cs.`payment_method`                 AS pg_type
FROM events_subscription_complete cs
LEFT JOIN bot_users b ON cs.user_id = b.user_id
WHERE cs.user_id IS NOT NULL AND cs.user_id <> ''
  AND STR_TO_DATE(cs.event_ts, '{DATE_FMT}') IS NOT NULL
  AND b.user_id IS NULL;


DROP VIEW IF EXISTS v_events_subscription_renew;
CREATE VIEW v_events_subscription_renew AS
SELECT
    rs.user_id,
    STR_TO_DATE(rs.event_ts, '{DATE_FMT}') AS event_time,
    rs.paid_amount,
    rs.`plan_price`              AS plan_price,
    rs.`discount_amount`  AS coupon_discount_amount,
    rs.`payment_method`                 AS pg_type
FROM events_subscription_renew rs
LEFT JOIN bot_users b ON rs.user_id = b.user_id
WHERE rs.user_id IS NOT NULL AND rs.user_id <> ''
  AND STR_TO_DATE(rs.event_ts, '{DATE_FMT}') IS NOT NULL
  AND b.user_id IS NULL;


DROP VIEW IF EXISTS v_events_subscription_resubscribe;
CREATE VIEW v_events_subscription_resubscribe AS
SELECT
    rss.user_id,
    STR_TO_DATE(rss.event_ts, '{DATE_FMT}') AS event_time,
    rss.paid_amount,
    rss.`plan_price`              AS plan_price,
    rss.`discount_amount`  AS coupon_discount_amount,
    rss.`payment_method`                 AS pg_type
FROM events_subscription_resubscribe rss
LEFT JOIN bot_users b ON rss.user_id = b.user_id
WHERE rss.user_id IS NOT NULL AND rss.user_id <> ''
  AND STR_TO_DATE(rss.event_ts, '{DATE_FMT}') IS NOT NULL
  AND b.user_id IS NULL;


DROP VIEW IF EXISTS v_events_trial_start;
CREATE VIEW v_events_trial_start AS
SELECT
    ft.user_id,
    STR_TO_DATE(ft.event_ts, '{DATE_FMT}') AS event_time,
    ft.`plan_price` AS plan_price,
    ft.`plan_type`  AS plan_type
FROM events_trial_start ft
LEFT JOIN bot_users b ON ft.user_id = b.user_id
WHERE ft.user_id IS NOT NULL AND ft.user_id <> ''
  AND STR_TO_DATE(ft.event_ts, '{DATE_FMT}') IS NOT NULL
  AND b.user_id IS NULL;
"""

execute_many(create_revenue_views_sql)

In [ ]:
run_query("""
SELECT 'v_events_payment_page_view'        AS view_name, COUNT(*) AS row_cnt FROM v_events_payment_page_view
UNION ALL SELECT 'v_events_subscription_complete',         COUNT(*) FROM v_events_subscription_complete
UNION ALL SELECT 'v_events_subscription_renew',            COUNT(*) FROM v_events_subscription_renew
UNION ALL SELECT 'v_events_subscription_resubscribe',      COUNT(*) FROM v_events_subscription_resubscribe
UNION ALL SELECT 'v_events_trial_start',              COUNT(*) FROM v_events_trial_start;
""", "revenue_view_check")

### 결제 퍼널 및 이탈 리스트

In [ ]:
query = '''
WITH
    signup AS (
        SELECT
            s.user_id,
            MIN(s.event_time) AS signup_time
        FROM v_events_signup s
        LEFT JOIN bot_users b
            ON s.user_id = b.user_id
        WHERE b.user_id IS NULL
        GROUP BY s.user_id
    ),

    first_content AS (
        SELECT
            sc.user_id,
            MIN(sc.event_time) AS first_content_time
        FROM v_events_content_start sc
        JOIN signup s
            ON sc.user_id = s.user_id
           AND sc.event_time >= s.signup_time
        GROUP BY sc.user_id
    ),

    first_lesson AS (
        SELECT
            el.user_id,
            MIN(el.event_time) AS first_lesson_time
        FROM v_events_lesson_view el
        JOIN first_content fc
            ON el.user_id = fc.user_id
           AND el.event_time >= fc.first_content_time
        GROUP BY el.user_id
    ),

    activation_users AS (
        SELECT
            cl.user_id,
            MIN(cl.event_time) AS activation_time
        FROM v_events_lesson_complete cl
        JOIN first_lesson fl
            ON cl.user_id = fl.user_id
           AND cl.event_time >= fl.first_lesson_time
        GROUP BY cl.user_id
    ),

    retained_users AS (
        SELECT
            a.user_id,
            MIN(el.event_time) AS retention_time
        FROM activation_users a
        JOIN v_events_lesson_view el
            ON a.user_id = el.user_id
           AND el.event_time >= DATE_ADD(a.activation_time, INTERVAL 24 HOUR)
           AND el.event_time <  DATE_ADD(a.activation_time, INTERVAL 8 DAY)
        GROUP BY a.user_id
    ),

    payment_page_users AS (
        SELECT
            r.user_id,
            MIN(pp.event_time) AS payment_page_time
        FROM retained_users r
        JOIN v_events_payment_page_view pp
            ON r.user_id = pp.user_id
           AND pp.event_time >= r.retention_time
        GROUP BY r.user_id
    ),

    subscribed_users AS (
        SELECT DISTINCT
            pp.user_id
        FROM payment_page_users pp
        JOIN v_events_subscription_complete cs
            ON pp.user_id = cs.user_id
           AND cs.event_time >= pp.payment_page_time
    )

SELECT
    COUNT(DISTINCT r.user_id) AS retention_users,
    COUNT(DISTINCT pp.user_id) AS payment_page_users,
    COUNT(DISTINCT s.user_id) AS subscribed_users,

    COUNT(DISTINCT r.user_id) - COUNT(DISTINCT pp.user_id) AS retention_to_payment_dropout_users,
    COUNT(DISTINCT pp.user_id) - COUNT(DISTINCT s.user_id) AS payment_to_subscription_dropout_users,

    ROUND(
        COUNT(DISTINCT pp.user_id) * 100.0 / NULLIF(COUNT(DISTINCT r.user_id), 0),
        2
    ) AS retention_to_payment_pct,

    ROUND(
        COUNT(DISTINCT s.user_id) * 100.0 / NULLIF(COUNT(DISTINCT pp.user_id), 0),
        2
    ) AS payment_to_subscription_pct,

    ROUND(
        COUNT(DISTINCT s.user_id) * 100.0 / NULLIF(COUNT(DISTINCT r.user_id), 0),
        2
    ) AS retention_to_subscription_pct

FROM retained_users r
LEFT JOIN payment_page_users pp
    ON r.user_id = pp.user_id
LEFT JOIN subscribed_users s
    ON pp.user_id = s.user_id;
'''

revenue_funnel_df = pd.read_sql(query, engine)
revenue_funnel_df

In [ ]:
import seaborn as sns

bar_colors = ['#bdd8f1', '#82a6cb', '#3667a6']
accent_color = '#214177'

stages = ["Retention", "Payment Page", "Subscribed"]
counts = [
    revenue_funnel_df['retention_users'].iloc[0],
    revenue_funnel_df['payment_page_users'].iloc[0],
    revenue_funnel_df['subscribed_users'].iloc[0]
]

total = counts[0]

plt.figure(figsize=(10, 6))
ax = sns.barplot(x=counts, y=stages, palette=bar_colors)

for i, count in enumerate(counts):
    pct = (count / total) * 100 if total > 0 else 0
    ax.text(
        count + total * 0.01,
        i,
        f'{count:,}명 ({pct:.1f}%)',
        va='center',
        fontsize=12,
        color=accent_color,
        fontweight='bold'
    )

plt.title('결제 퍼널 및 이탈 현황', fontsize=15, fontweight='bold')
plt.xlabel('Number of Users')
plt.ylabel('')
plt.xlim(0, max(counts) * 1.2)
plt.grid(axis='x', linestyle='--', alpha=0.3)

for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)

plt.tight_layout()
plt.show()

### 구독 요금 구조

In [ ]:
query = '''
WITH
    signup AS (
        SELECT
            s.user_id,
            MIN(s.event_time) AS signup_time
        FROM v_events_signup s
        LEFT JOIN bot_users b ON s.user_id = b.user_id
        WHERE b.user_id IS NULL
        GROUP BY s.user_id
    ),

    first_content AS (
        SELECT
            sc.user_id,
            MIN(sc.event_time) AS first_content_time
        FROM v_events_content_start sc
        JOIN signup s
            ON sc.user_id = s.user_id
           AND sc.event_time >= s.signup_time
        GROUP BY sc.user_id
    ),

    first_lesson AS (
        SELECT
            el.user_id,
            MIN(el.event_time) AS first_lesson_time
        FROM v_events_lesson_view el
        JOIN first_content fc
            ON el.user_id = fc.user_id
           AND el.event_time >= fc.first_content_time
        GROUP BY el.user_id
    ),

    activation_users AS (
        SELECT
            cl.user_id,
            MIN(cl.event_time) AS activation_time
        FROM v_events_lesson_complete cl
        JOIN first_lesson fl
            ON cl.user_id = fl.user_id
           AND cl.event_time >= fl.first_lesson_time
        GROUP BY cl.user_id
    ),

    retained_users AS (
        SELECT DISTINCT a.user_id
        FROM activation_users a
        JOIN v_events_lesson_view el
            ON a.user_id = el.user_id
           AND el.event_time >= DATE_ADD(a.activation_time, INTERVAL 24 HOUR)
           AND el.event_time <  DATE_ADD(a.activation_time, INTERVAL 8 DAY)
    ),

    subscription_events AS (
        SELECT
            cs.user_id,
            cs.event_time,
            cs.plan_price,
            cs.paid_amount,
            cs.coupon_discount_amount,
            cs.pg_type,
            ROW_NUMBER() OVER (
                PARTITION BY cs.user_id
                ORDER BY cs.event_time
            ) AS rn
        FROM v_events_subscription_complete cs
        JOIN retained_users r ON cs.user_id = r.user_id
    ),

    first_subscription AS (
        SELECT
            user_id,
            event_time,
            plan_price,
            paid_amount,
            coupon_discount_amount,
            pg_type
        FROM subscription_events
        WHERE rn = 1
    )

SELECT
    plan_price,
    COUNT(DISTINCT user_id) AS subscribed_users,

    ROUND(
        COUNT(DISTINCT user_id) * 100.0
        / NULLIF(SUM(COUNT(DISTINCT user_id)) OVER (), 0),
        2
    ) AS user_share_pct,

    SUM(paid_amount) AS total_revenue,
    ROUND(AVG(paid_amount), 0) AS avg_paid_amount,

    COUNT(DISTINCT CASE
        WHEN coupon_discount_amount > 0 THEN user_id
    END) AS coupon_used_users,

    ROUND(
        COUNT(DISTINCT CASE
            WHEN coupon_discount_amount > 0 THEN user_id
        END) * 100.0 / NULLIF(COUNT(DISTINCT user_id), 0),
        2
    ) AS coupon_used_pct,

    ROUND(
        AVG(CASE
            WHEN coupon_discount_amount > 0 THEN coupon_discount_amount
        END),
        0
    ) AS avg_discount_amount

FROM first_subscription
GROUP BY plan_price
ORDER BY subscribed_users DESC;
'''

df_plan = pd.read_sql(query, engine)
df_plan

In [ ]:
query = '''
WITH
    signup AS (
        SELECT
            s.user_id,
            MIN(s.event_time) AS signup_time
        FROM v_events_signup s
        LEFT JOIN bot_users b ON s.user_id = b.user_id
        WHERE b.user_id IS NULL
        GROUP BY s.user_id
    ),

    first_content AS (
        SELECT
            sc.user_id,
            MIN(sc.event_time) AS first_content_time
        FROM v_events_content_start sc
        JOIN signup s
            ON sc.user_id = s.user_id
           AND sc.event_time >= s.signup_time
        GROUP BY sc.user_id
    ),

    first_lesson AS (
        SELECT
            el.user_id,
            MIN(el.event_time) AS first_lesson_time
        FROM v_events_lesson_view el
        JOIN first_content fc
            ON el.user_id = fc.user_id
           AND el.event_time >= fc.first_content_time
        GROUP BY el.user_id
    ),

    activation_users AS (
        SELECT
            cl.user_id,
            MIN(cl.event_time) AS activation_time
        FROM v_events_lesson_complete cl
        JOIN first_lesson fl
            ON cl.user_id = fl.user_id
           AND cl.event_time >= fl.first_lesson_time
        GROUP BY cl.user_id
    ),

    retained_users AS (
        SELECT
            a.user_id,
            MIN(el.event_time) AS retention_time
        FROM activation_users a
        JOIN v_events_lesson_view el
            ON a.user_id = el.user_id
           AND el.event_time >= DATE_ADD(a.activation_time, INTERVAL 24 HOUR)
           AND el.event_time <  DATE_ADD(a.activation_time, INTERVAL 8 DAY)
        GROUP BY a.user_id
    ),

    subscription_events AS (
        SELECT
            cs.user_id,
            cs.event_time,
            cs.plan_price,
            cs.paid_amount,
            cs.coupon_discount_amount,
            cs.pg_type,
            ROW_NUMBER() OVER (
                PARTITION BY cs.user_id
                ORDER BY cs.event_time
            ) AS rn
        FROM v_events_subscription_complete cs
        JOIN retained_users r
            ON cs.user_id = r.user_id
           AND cs.event_time >= r.retention_time
    ),

    first_subscription AS (
        SELECT
            user_id,
            event_time,
            plan_price,
            paid_amount,
            coupon_discount_amount,
            pg_type
        FROM subscription_events
        WHERE rn = 1
    )

SELECT
    plan_price,
    COUNT(DISTINCT user_id) AS subscribed_users,

    ROUND(
        COUNT(DISTINCT user_id) * 100.0
        / NULLIF(SUM(COUNT(DISTINCT user_id)) OVER (), 0),
        2
    ) AS user_share_pct,

    SUM(paid_amount) AS total_revenue,
    ROUND(AVG(paid_amount), 0) AS avg_paid_amount,

    COUNT(DISTINCT CASE
        WHEN coupon_discount_amount > 0 THEN user_id
    END) AS coupon_used_users,

    ROUND(
        COUNT(DISTINCT CASE
            WHEN coupon_discount_amount > 0 THEN user_id
        END) * 100.0 / NULLIF(COUNT(DISTINCT user_id), 0),
        2
    ) AS coupon_used_pct,

    ROUND(
        AVG(CASE
            WHEN coupon_discount_amount > 0 THEN coupon_discount_amount
        END),
        0
    ) AS avg_discount_amount

FROM first_subscription
GROUP BY plan_price
ORDER BY subscribed_users DESC;
'''

df_plan = pd.read_sql(query, engine)
df_plan

### MRR & ARR 지표

#### 전체 유저

In [ ]:
query = '''
WITH
    all_revenue AS (
        SELECT user_id, paid_amount, event_time
        FROM v_events_subscription_complete
        UNION ALL
        SELECT user_id, paid_amount, event_time
        FROM v_events_subscription_renew
        UNION ALL
        SELECT user_id, paid_amount, event_time
        FROM v_events_subscription_resubscribe
    ),

    revenue_with_plan AS (
        SELECT
            ar.user_id,
            ar.paid_amount,
            ar.event_time,
            ft.plan_type
        FROM all_revenue ar
        JOIN v_events_trial_start ft ON ar.user_id = ft.user_id
    ),

    monthly_revenue AS (
        SELECT
            DATE_FORMAT(event_time, '%%Y-%%m') AS revenue_month,
            SUM(CASE
                WHEN plan_type = @ANNUAL_PLAN THEN paid_amount / 12
                WHEN plan_type = @MONTHLY_PLAN  THEN paid_amount
                ELSE paid_amount
            END) AS calculated_mrr
        FROM revenue_with_plan
        GROUP BY 1
    )
SELECT
    revenue_month,
    ROUND(calculated_mrr, 0)      AS MRR_TOTAL,
    ROUND(calculated_mrr * 12, 0) AS ARR_TOTAL
FROM monthly_revenue
ORDER BY 1
'''

all_mrr_arr_df = pd.read_sql(query, engine)
all_mrr_arr_df

In [ ]:
query_all = '''
WITH
    all_revenue AS (
        SELECT user_id, paid_amount, event_time
        FROM v_events_subscription_complete
        UNION ALL
        SELECT user_id, paid_amount, event_time
        FROM v_events_subscription_renew
        UNION ALL
        SELECT user_id, paid_amount, event_time
        FROM v_events_subscription_resubscribe
    ),

    user_plan AS (
        SELECT user_id, MAX(plan_type) AS plan_type
        FROM v_events_trial_start
        GROUP BY user_id
    ),

    revenue_with_plan AS (
        SELECT
            ar.user_id,
            ar.paid_amount,
            ar.event_time,
            up.plan_type
        FROM all_revenue ar
        LEFT JOIN user_plan up ON ar.user_id = up.user_id
    ),

    monthly_revenue AS (
        SELECT
            DATE_FORMAT(event_time, '%%Y-%%m') AS revenue_month,
            SUM(CASE
                WHEN plan_type = @ANNUAL_PLAN  THEN paid_amount / 12
                WHEN plan_type = @MONTHLY_PLAN THEN paid_amount
                -- plan_type이 NULL인 경우 결제 금액 기준으로 연간 플랜 판별
                WHEN paid_amount >= @ANNUAL_PLAN_THRESHOLD THEN paid_amount / 12
                ELSE paid_amount
            END) AS calculated_mrr
        FROM revenue_with_plan
        GROUP BY 1
    )

SELECT
    revenue_month,
    ROUND(calculated_mrr, 0)      AS MRR_TOTAL,
    ROUND(calculated_mrr * 12, 0) AS ARR_TOTAL
FROM monthly_revenue
ORDER BY 1
'''

all_mrr_arr_df = pd.read_sql(query_all, engine)
all_mrr_arr_df

####  Revenue 유저

In [ ]:
query_retention = '''
WITH
    signup AS (
        SELECT s.user_id, MIN(s.event_time) AS signup_time
        FROM v_events_signup s
        LEFT JOIN bot_users b ON s.user_id = b.user_id
        WHERE b.user_id IS NULL
        GROUP BY s.user_id
    ),
    first_content AS (
        SELECT sc.user_id, MIN(sc.event_time) AS first_content_time
        FROM v_events_content_start sc
        JOIN signup s
            ON sc.user_id = s.user_id
           AND sc.event_time >= s.signup_time
        GROUP BY sc.user_id
    ),
    first_lesson AS (
        SELECT el.user_id, MIN(el.event_time) AS first_lesson_time
        FROM v_events_lesson_view el
        JOIN first_content fc
            ON el.user_id = fc.user_id
           AND el.event_time >= fc.first_content_time
        GROUP BY el.user_id
    ),
    activation_users AS (
        SELECT cl.user_id, MIN(cl.event_time) AS activation_time
        FROM v_events_lesson_complete cl
        JOIN first_lesson fl
            ON cl.user_id = fl.user_id
           AND cl.event_time >= fl.first_lesson_time
        GROUP BY cl.user_id
    ),
    retained_users AS (
        SELECT DISTINCT a.user_id
        FROM activation_users a
        JOIN v_events_lesson_view el
            ON a.user_id = el.user_id
           AND el.event_time >= DATE_ADD(a.activation_time, INTERVAL 24 HOUR)
           AND el.event_time <  DATE_ADD(a.activation_time, INTERVAL 8 DAY)
    ),

    all_revenue AS (
        SELECT cs.user_id, cs.paid_amount, cs.event_time
        FROM v_events_subscription_complete cs
        JOIN retained_users r ON cs.user_id = r.user_id

        UNION ALL

        SELECT rs.user_id, rs.paid_amount, rs.event_time
        FROM v_events_subscription_renew rs
        JOIN retained_users r ON rs.user_id = r.user_id

        UNION ALL

        SELECT rss.user_id, rss.paid_amount, rss.event_time
        FROM v_events_subscription_resubscribe rss
        JOIN retained_users r ON rss.user_id = r.user_id
    ),

    user_plan AS (
        SELECT user_id, MAX(plan_type) AS plan_type
        FROM v_events_trial_start
        GROUP BY user_id
    ),

    revenue_with_plan AS (
        SELECT
            ar.user_id,
            ar.paid_amount,
            ar.event_time,
            up.plan_type
        FROM all_revenue ar
        LEFT JOIN user_plan up ON ar.user_id = up.user_id
    ),

    monthly_revenue AS (
        SELECT
            DATE_FORMAT(event_time, '%%Y-%%m') AS revenue_month,
            SUM(CASE
                WHEN plan_type = @ANNUAL_PLAN THEN paid_amount / 12
                WHEN plan_type = @MONTHLY_PLAN  THEN paid_amount
                WHEN paid_amount >= @ANNUAL_PLAN_THRESHOLD THEN paid_amount / 12
                ELSE paid_amount
            END) AS calculated_mrr
        FROM revenue_with_plan
        GROUP BY 1
    )

SELECT
    revenue_month,
    ROUND(calculated_mrr, 0) AS MRR_REV,
    ROUND(calculated_mrr * 12, 0) AS ARR_REV
FROM monthly_revenue
ORDER BY 1
'''

mrr_arr_df = pd.read_sql(query_retention, engine)
mrr_arr_df

#### 시각화

In [ ]:
import numpy as np


compare_df = pd.merge(
    all_mrr_arr_df[['revenue_month', 'MRR_TOTAL', 'ARR_TOTAL']],
    mrr_arr_df[['revenue_month', 'MRR_REV', 'ARR_REV']],
    on='revenue_month',
    how='left',
    suffixes=('_Total', '_Retention')
)


fig, ax1 = plt.subplots(figsize=(14, 7))


x = np.arange(len(compare_df['revenue_month']))
width = 0.35


ax1.bar(x - width/2, compare_df['MRR_TOTAL'], width,
        label='Total MRR (전체)', color='#82a6cb', alpha=0.7)
ax1.bar(x + width/2, compare_df['MRR_REV'], width,
        label='Revenue MRR (Revenue 유저)', color='#214177', alpha=0.7)

ax1.set_xlabel('매출 월 (Month)', fontsize=12)
ax1.set_ylabel('MRR (Monthly Recurring Revenue)', fontsize=12)
ax1.set_xticks(x)
ax1.set_xticklabels(compare_df['revenue_month'], rotation=45)


ax2 = ax1.twinx()
ax2.plot(x, compare_df['ARR_TOTAL'],
         label='Total ARR (전체)', color='#82a6cb',
         marker='o', linewidth=3, markersize=7)
ax2.plot(x, compare_df['ARR_REV'],
         label='Revenue ARR (Revenue 유저)', color='#214177',
         marker='s', linewidth=3, markersize=7)

ax2.set_ylabel('ARR (Annualized Run Rate)', fontsize=12)


y1_min, y1_max = ax1.get_ylim()
ax2.set_ylim(y1_min * 12, y1_max * 12)


ax1.legend(loc='upper left', fontsize=11)
ax2.legend(loc='upper right', fontsize=11)

ax1.spines[['top', 'right', 'left']].set_visible(False)
ax2.spines[['top', 'right', 'left']].set_visible(False)

ax1.grid(axis='y', linestyle='--', alpha=0.3)
ax2.grid(axis='y', linestyle='--', alpha=0.3)

plt.title('MRR & ARR 지표: 전체 매출 vs Revenue 유저',
          fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

In [ ]:
compare_df = pd.merge(
    all_mrr_arr_df[['revenue_month', 'MRR_TOTAL']],
    mrr_arr_df[['revenue_month', 'MRR_REV']],
    on='revenue_month',
    how='left',
    suffixes=('_Total', '_Retention')
)

fig, ax = plt.subplots(figsize=(12, 6))

sns.lineplot(x='revenue_month', y='MRR_TOTAL', data=compare_df,
             ax=ax, color='#bdd8f1', marker='o', linewidth=3,
             label='Total MRR (전체 매출)')

sns.lineplot(x='revenue_month', y='MRR_REV', data=compare_df,
             ax=ax, color='#3667a6', marker='s', linewidth=3,
             label='Retention MRR (Revenue 유저 매출)')

ax.fill_between(compare_df['revenue_month'],
                compare_df['MRR_REV'],
                compare_df['MRR_TOTAL'],
                color='gray', alpha=0.15, label='매출 갭')

ax.spines[['top', 'right', 'left']].set_visible(False)
ax.grid(axis='y', linestyle='--', alpha=0.3)

plt.title('전체 매출 vs Revenue 유저 매출 갭', fontsize=16, fontweight='bold')
plt.xlabel('매출 월 (Month)', fontsize=12)
plt.ylabel('MRR (단위: 원)', fontsize=12)
plt.xticks(rotation=45)
plt.legend(fontsize=12)
plt.tight_layout()
plt.show()

### ROAS

#### 전체

In [ ]:
query = '''
WITH
    all_revenue AS (
        SELECT cs.user_id, cs.paid_amount
        FROM v_events_subscription_complete cs
        LEFT JOIN bot_users b ON cs.user_id = b.user_id
        WHERE cs.user_id IS NOT NULL AND cs.user_id <> ''
          AND b.user_id IS NULL

        UNION ALL

        SELECT rs.user_id, rs.paid_amount
        FROM v_events_subscription_renew rs
        LEFT JOIN bot_users b ON rs.user_id = b.user_id
        WHERE rs.user_id IS NOT NULL AND rs.user_id <> ''
          AND b.user_id IS NULL

        UNION ALL

        SELECT rss.user_id, rss.paid_amount
        FROM v_events_subscription_resubscribe rss
        LEFT JOIN bot_users b ON rss.user_id = b.user_id
        WHERE rss.user_id IS NOT NULL AND rss.user_id <> ''
          AND b.user_id IS NULL
    ),

    channel_revenue AS (
        SELECT
            ua.last_channel,
            SUM(ar.paid_amount) AS total_revenue
        FROM all_revenue ar
        JOIN user_acquisition ua ON ar.user_id = ua.user_id
        GROUP BY ua.last_channel
    ),

    channel_spend AS (
        SELECT
            channel,
            SUM(spend_krw) AS total_spend
        FROM marketing_spend_daily
        GROUP BY channel
    )

SELECT
    s.channel,
    s.total_spend AS ad_spend,
    COALESCE(r.total_revenue, 0) AS revenue,
    ROUND((COALESCE(r.total_revenue, 0) / NULLIF(s.total_spend, 0)) * 100, 2) AS ROAS_percent
FROM channel_spend s
LEFT JOIN channel_revenue r ON s.channel = r.last_channel
ORDER BY ROAS_percent DESC
'''

df_roas_overall = pd.read_sql(query, engine)
df_roas_overall

#### Revenue 유저

In [ ]:
query = '''
WITH
    valid_users AS (
        SELECT user_id FROM v_events_signup
        WHERE user_id IS NOT NULL AND user_id <> ''
          AND user_id NOT IN (SELECT user_id FROM bot_users)
    ),

    signup AS (
        SELECT s.user_id, MIN(s.event_time) AS signup_time
        FROM v_events_signup s
        INNER JOIN valid_users vu ON s.user_id = vu.user_id
        GROUP BY s.user_id
    ),

    first_content AS (
        SELECT sc.user_id, MIN(sc.event_time) AS first_content_time
        FROM v_events_content_start sc
        INNER JOIN signup s ON sc.user_id = s.user_id AND sc.event_time >= s.signup_time
        GROUP BY sc.user_id
    ),

    first_lesson AS (
        SELECT el.user_id, MIN(el.event_time) AS first_lesson_time
        FROM v_events_lesson_view el
        INNER JOIN first_content fc ON el.user_id = fc.user_id AND el.event_time >= fc.first_content_time
        GROUP BY el.user_id
    ),

    activation_users AS (
        SELECT cl.user_id, MIN(cl.event_time) AS activation_time
        FROM v_events_lesson_complete cl
        INNER JOIN first_lesson fl ON cl.user_id = fl.user_id AND cl.event_time >= fl.first_lesson_time
        GROUP BY cl.user_id
    ),

    retained_users AS (
        SELECT a.user_id, MIN(el.event_time) AS retention_time
        FROM activation_users a
        JOIN v_events_lesson_view el ON a.user_id = el.user_id
           AND el.event_time >= DATE_ADD(a.activation_time, INTERVAL 24 HOUR)
           AND el.event_time <  DATE_ADD(a.activation_time, INTERVAL 8 DAY)
        GROUP BY a.user_id
    ),

    payment_page_users AS (
        SELECT p.user_id, MIN(p.event_time) AS payment_time
        FROM v_events_payment_page_view p
        INNER JOIN retained_users r ON p.user_id = r.user_id
           AND p.event_time >= r.retention_time
        GROUP BY p.user_id
    ),

    all_revenue AS (
        SELECT user_id, paid_amount, event_time FROM v_events_subscription_complete
        UNION ALL
        SELECT user_id, paid_amount, event_time FROM v_events_subscription_renew
        UNION ALL
        SELECT user_id, paid_amount, event_time FROM v_events_subscription_resubscribe
    ),

    revenue_users AS (
        SELECT ar.user_id, SUM(ar.paid_amount) AS total_revenue
        FROM all_revenue ar
        INNER JOIN payment_page_users pp ON ar.user_id = pp.user_id
           AND ar.event_time >= pp.payment_time
        GROUP BY ar.user_id
    ),

    channel_revenue AS (
        SELECT
            ua.last_channel,
            SUM(ru.total_revenue) AS total_revenue
        FROM revenue_users ru
        JOIN user_acquisition ua ON ru.user_id = ua.user_id
        GROUP BY ua.last_channel
    ),

    channel_spend AS (
        SELECT
            channel,
            SUM(spend_krw) AS total_spend
        FROM marketing_spend_daily
        GROUP BY channel
    )

SELECT
    s.channel,
    s.total_spend AS ad_spend,
    COALESCE(r.total_revenue, 0) AS revenue,
    ROUND((COALESCE(r.total_revenue, 0) / NULLIF(s.total_spend, 0)) * 100, 2) AS ROAS_percent
FROM channel_spend s
LEFT JOIN channel_revenue r ON s.channel = r.last_channel
ORDER BY ROAS_percent DESC
'''

df_roas_revenue_funnel = pd.read_sql(query, engine)
df_roas_revenue_funnel

#### 시각화

In [ ]:
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

channels = ['Email', 'Paid Search', 'Social Paid']
ad_spend = [164314549, 2872670939, 2841523894]
revenue = [7205804, 26746341, 21311691]
roas_percent = [4.39, 0.93, 0.75]

x = np.arange(len(channels))
width = 0.35

fig, ax1 = plt.subplots(figsize=(10, 6))
ax2 = ax1.twinx()

rects1 = ax1.bar(x - width/2, ad_spend, width, label='광고비 지출액', color='#C6D8EE', edgecolor='white')
rects2 = ax1.bar(x + width/2, revenue, width, label='실제 수익 (Revenue)', color='#4F6EAA', edgecolor='white')

line = ax2.plot(x, roas_percent, color='#E97132', marker='o', linewidth=2, markersize=8, label='ROAS (%)')

ax1.set_ylabel('금액 (원)', labelpad=15)
ax2.set_ylabel('ROAS (%)', labelpad=15)
ax1.set_xticks(x)
ax1.set_xticklabels(channels, fontsize=12)

ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda val, loc: "{:,}".format(int(val))))

ax1.yaxis.grid(True, linestyle='-', which='major', color='lightgrey', alpha=0.8)
ax1.set_axisbelow(True)

for i, v in enumerate(roas_percent):
    ax2.annotate(f'{v}%', xy=(i, v), xytext=(0, 10), textcoords="offset points", ha='center', va='bottom', color='#E97132', fontweight='bold')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper center', bbox_to_anchor=(0.5, 1.15), ncol=3, frameon=False)

for ax in [ax1, ax2]:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False) if ax == ax1 else ax.spines['left'].set_visible(False)

plt.tight_layout()
plt.show()

### ARPU(유/무료 고객 전체 평균 매출)-ARPPU(유료 고객 평균 매출)

#### 전체

In [ ]:
query = '''
WITH
    all_revenue AS (
        SELECT cs.user_id, cs.paid_amount
        FROM v_events_subscription_complete cs
        LEFT JOIN bot_users b ON cs.user_id = b.user_id
        WHERE cs.user_id IS NOT NULL AND cs.user_id <> ''
          AND b.user_id IS NULL

        UNION ALL

        SELECT rs.user_id, rs.paid_amount
        FROM v_events_subscription_renew rs
        LEFT JOIN bot_users b ON rs.user_id = b.user_id
        WHERE rs.user_id IS NOT NULL AND rs.user_id <> ''
          AND b.user_id IS NULL

        UNION ALL

        SELECT rss.user_id, rss.paid_amount
        FROM v_events_subscription_resubscribe rss
        LEFT JOIN bot_users b ON rss.user_id = b.user_id
        WHERE rss.user_id IS NOT NULL AND rss.user_id <> ''
          AND b.user_id IS NULL
    ),

    total_users AS (
        SELECT COUNT(DISTINCT s.user_id) AS total_user_count
        FROM v_events_signup s
        LEFT JOIN bot_users b ON s.user_id = b.user_id
        WHERE s.user_id IS NOT NULL AND s.user_id <> ''
          AND b.user_id IS NULL
    ),

    revenue_summary AS (
        SELECT
            COUNT(DISTINCT user_id) AS pu_count,
            SUM(paid_amount)      AS total_revenue
        FROM all_revenue
    )

SELECT
    tu.total_user_count AS total_users,
    rs.pu_count AS paying_users,
    rs.total_revenue AS total_revenue,
    ROUND(rs.total_revenue / tu.total_user_count, 0) AS ARPU,
    ROUND(rs.total_revenue / rs.pu_count, 0) AS ARPPU
FROM total_users tu
CROSS JOIN revenue_summary rs
'''

df_overall_metrics = pd.read_sql(query, engine)
df_overall_metrics

#### Revenue 유저

In [ ]:
query = '''
WITH
    valid_users AS (
        SELECT user_id FROM v_events_signup
        WHERE user_id IS NOT NULL AND user_id NOT IN (SELECT user_id FROM bot_users)
    ),

    signup AS (
        SELECT s.user_id, MIN(s.event_time) AS signup_time
        FROM v_events_signup s
        INNER JOIN valid_users vu ON s.user_id = vu.user_id
        GROUP BY s.user_id
    ),

    first_content AS (
        SELECT sc.user_id, MIN(sc.event_time) AS first_content_time
        FROM v_events_content_start sc
        INNER JOIN signup s ON sc.user_id = s.user_id AND sc.event_time >= s.signup_time
        GROUP BY sc.user_id
    ),

    first_lesson AS (
        SELECT el.user_id, MIN(el.event_time) AS first_lesson_time
        FROM v_events_lesson_view el
        INNER JOIN first_content fc ON el.user_id = fc.user_id AND el.event_time >= fc.first_content_time
        GROUP BY el.user_id
    ),

    activation_users AS (
        SELECT cl.user_id, MIN(cl.event_time) AS activation_time
        FROM v_events_lesson_complete cl
        INNER JOIN first_lesson fl ON cl.user_id = fl.user_id AND cl.event_time >= fl.first_lesson_time
        GROUP BY cl.user_id
    ),

    retained_users AS (
        SELECT a.user_id, MIN(el.event_time) AS retention_time
        FROM activation_users a
        JOIN v_events_lesson_view el ON a.user_id = el.user_id
           AND el.event_time >= DATE_ADD(a.activation_time, INTERVAL 24 HOUR)
           AND el.event_time <  DATE_ADD(a.activation_time, INTERVAL 8 DAY)
        GROUP BY a.user_id
    ),

    payment_page_users AS (
        SELECT p.user_id, MIN(p.event_time) AS payment_time
        FROM v_events_payment_page_view p
        INNER JOIN retained_users r ON p.user_id = r.user_id
           AND p.event_time >= r.retention_time
        GROUP BY p.user_id
    ),

    revenue_users AS (
        SELECT cs.user_id, SUM(cs.paid_amount) AS paid_amount
        FROM v_events_subscription_complete cs
        INNER JOIN payment_page_users pp ON cs.user_id = pp.user_id
           AND cs.event_time >= pp.payment_time
        GROUP BY cs.user_id
    )

SELECT
    (SELECT COUNT(*) FROM retained_users) AS total_retention_users,
    (SELECT COUNT(*) FROM revenue_users)  AS paying_users,
    (SELECT SUM(paid_amount) FROM revenue_users) AS total_revenue,

    ROUND((SELECT SUM(paid_amount) FROM revenue_users) / (SELECT COUNT(*) FROM retained_users), 0) AS ARPU_REV,
    ROUND((SELECT SUM(paid_amount) FROM revenue_users) / (SELECT COUNT(*) FROM revenue_users), 0) AS ARPPU_REV

'''

df_rev_metrics = pd.read_sql(query, engine)
df_rev_metrics

#### 시각화

In [ ]:
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

labels = ['ARPU', 'ARPPU']
all_users = [11865, 65129]
rev_users = [2705, 44352]

x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 6))

rects1 = ax.bar(x - width/2, all_users, width, label='전체 유저', color='#C6D8EE', edgecolor='white')
rects2 = ax.bar(x + width/2, rev_users, width, label='Revenue 유저', color='#4F6EAA', edgecolor='white')

ax.set_ylabel('금액 (원)', labelpad=15)
ax.set_ylim(0, 90000)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, loc: "{:,}".format(int(x))))

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=12)

ax.legend(loc='upper center', bbox_to_anchor=(0.5, 1.1), ncol=2, frameon=False, fontsize=11, handlelength=1.2)

ax.yaxis.grid(True, linestyle='-', which='major', color='lightgrey', alpha=0.8)
ax.set_axisbelow(True)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False)

def autolabel(rects, text_color):
    for rect in rects:
        height = rect.get_height()

        if height < 8000:
            ax.annotate(f'{int(height):,}',
                        xy=(rect.get_x() + rect.get_width() / 2, height),
                        xytext=(0, 5),
                        textcoords="offset points",
                        ha='center', va='bottom', color='black', fontsize=11)
        else:
            ax.annotate(f'{int(height):,}',
                        xy=(rect.get_x() + rect.get_width() / 2, height),
                        xytext=(0, -20),
                        textcoords="offset points",
                        ha='center', va='top', color=text_color, fontsize=11)

autolabel(rects1, 'black')
autolabel(rects2, 'white')

plt.tight_layout(rect=[0, 0.08, 1, 1])
plt.show()